In [1]:
#%matplotlib ipympl

import matplotlib.pyplot as plt
import netCDF4 as nc4
import xarray as xr
import numpy as np

import ls2d

from microhhpy.io import xr_open_groups
from microhhpy.thermo import exner

In [2]:
def get_cloud_bounds(ds, threshold=1e-8):
    cloud_mask = ds.qlqi > threshold

    nt = ds.sizes['time']
    z = ds.z.values

    z_bot = np.full(nt, np.nan)
    z_top = np.full(nt, np.nan)

    for t in range(nt):
        idx = np.where(cloud_mask[t].values)[0]
        if len(idx) > 0:
            z_bot[t] = z[idx[0]]
            z_top[t] = z[idx[-1]]

    ds['qlqi_bot'] = ('time', z_bot)
    ds['qlqi_top'] = ('time', z_top)


def read_stat(f, averaging_period, profile_mean):
    # Read all NetCDF groups.
    ds = xr_open_groups(f)

    # Average in time.
    dt = float(ds.time[1] - ds.time[0])
    n = int(averaging_period / dt)
    dsc = ds.coarsen(time=n, boundary='trim').mean()

    # Calculate mean, clipping end time to available data.
    t_end = min(profile_mean[1], float(ds.time[-1]))
    dsm = ds.sel(time=slice(profile_mean[0], t_end)).mean(dim='time')

    # Get cloud bottom/top.
    get_cloud_bounds(dsc)

    return dsc, dsm

In [3]:
path = '/home/scratch2/bart/microhh/mock_walker/grid_tests_snellius'

averaging_period =  43200
profile_mean = (20*24*3600, 30*24*3600)

ds_144, dsm_144 = read_stat(f'{path}/512_200_144/rcemip_ii.default.0000000.nc', averaging_period, profile_mean)
ds_128, dsm_128 = read_stat(f'{path}/512_200_128l/rcemip_ii.default.0000000.nc', averaging_period, profile_mean)

ds_100, dsm_100 = read_stat(f'{path}/1024_100_128h/rcemip_ii.default.nc', averaging_period, profile_mean)
ds_200, dsm_200 = read_stat(f'{path}/512_200_128h/rcemip_ii.default.0000000.nc', averaging_period, profile_mean)
ds_400, dsm_400 = read_stat(f'{path}/256_400_128h/rcemip_ii.default.0000000.nc', averaging_period, profile_mean)
ds_800, dsm_800 = read_stat(f'{path}/128_800_128h/rcemip_ii.default.0000000.nc', averaging_period, profile_mean)

In [4]:
def plot_time_series(runs):
    """
    Time series.
    """

    plt.figure(figsize=(8, 7), layout='constrained')
    
    plt.subplot(331)
    for ds, _, label, color, ls in runs:
        plt.plot(ds.time/3600/24, ds.ql_cover, label=label, color=color, ls=ls)
    plt.legend()
    plt.ylabel('cloud cover (liquid) (-)')
    
    plt.subplot(332)
    for ds, _, _, color, ls in runs:
        plt.plot(ds.time/3600/24, ds.qi_cover, color=color, ls=ls)
    plt.ylabel('cloud cover (ice) (-)')
    
    plt.subplot(333)
    for ds, _, _, color, ls in runs:
        plt.plot(ds.time/3600/24, ds.ql_path + ds.qi_path, color=color, ls=ls)
    plt.ylabel('LWP (kg m$^{-2}$)')
    plt.xlabel('Time (days)')
    plt.ylim(0, 0.03)
    
    plt.subplot(334)
    for ds, _, _, color, ls in runs:
        plt.plot(ds.time/3600/24, ds.qr_path, color=color, ls=ls)
    plt.ylabel('RWP (kg m$^{-2}$)')
    plt.xlabel('Time (days)')
    plt.ylim(0, 0.04)

    plt.subplot(335)
    for ds, _, _, color, ls in runs:
        plt.plot(ds.time/3600/24, ds.rr, color=color, ls=ls)
    plt.ylabel('rain rate (mm s$^{-1}$)')
    plt.xlabel('Time (days)')
    #plt.ylim(0, 0.04)
    
    plt.subplot(336)
    for ds, _, _, color, ls in runs:
        plt.plot(ds.time/3600/24, ds.sw_flux_dn[:,0], color=color, ls=ls)
    plt.ylabel('sw_dn_sfc (W m$^{-2}$)')
    plt.xlabel('Time (days)')
    plt.ylim(230, 300)
    
    plt.subplot(337)
    for ds, _, _, color, ls in runs:
        plt.plot(ds.time/3600/24, ds.lw_flux_up[:,-1], color=color, ls=ls)
    plt.ylabel('lw_dn_tod (W m$^{-2}$)')
    plt.xlabel('Time (days)')
    plt.ylim(220, 280)
    
    plt.subplot(338)
    for ds, _, _, color, ls in runs:
        plt.plot(ds.time/3600/24, ds.qlqi_bot, color=color, ls=ls)
    plt.ylabel('Cloud base (m)')
    plt.xlabel('Time (days)')
    
    plt.subplot(339)
    for ds, _, _, color, ls in runs:
        plt.plot(ds.time/3600/24, ds.qlqi_top, color=color, ls=ls)
    plt.ylabel('Cloud top (m)')
    plt.xlabel('Time (days)')
    plt.ylim(10000, 16000)

In [5]:
def plot_profiles(runs):
    """
    Profiles.
    """
    ylim = (0, 17000)

    plt.figure(figsize=(8, 7), layout='constrained')

    plt.subplot(331)
    for _, ds, label, color, ls in runs:
        plt.plot(ds.thl, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$\theta_l$ (K)')
    plt.ylabel('z (m)')
    plt.ylim(ylim)
    plt.xlim(283, 403)

    plt.subplot(332)
    for _, ds, label, color, ls in runs:
        plt.plot(ds.qt*1000, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$q_t$ (g/kg)')
    plt.ylabel('z (m)')
    plt.ylim(ylim)

    plt.subplot(333)
    for _, ds, label, color, ls in runs:
        plt.plot(ds.ql*1000, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$q_l$ (g/kg)')
    plt.ylabel('z (m)')
    plt.ylim(ylim)

    plt.subplot(334)
    for _, ds, label, color, ls in runs:
        plt.plot(ds.qi*1000, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$q_i$ (g/kg)')
    plt.ylabel('z (m)')
    plt.ylim(ylim)

    plt.subplot(335)
    for _, ds, label, color, ls in runs:
        plt.plot(ds.qr*1000, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$q_r$ (g/kg)')
    plt.ylim(ylim)

    plt.subplot(336)
    for _, ds, label, color, ls in runs:
        plt.plot(ds.qs*1000, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$q_s$ (g/kg)')
    plt.ylim(ylim)

    plt.subplot(337)
    for _, ds, label, color, ls in runs:
        plt.plot(ds.sw_flux_dn, ds.zh, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$S\downarrow$ (W/m2)')
    plt.ylabel('z (m)')
    plt.ylim(ylim)

    plt.subplot(338)
    for _, ds, label, color, ls in runs:
        plt.plot(ds.lw_flux_dn, ds.zh, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$L\downarrow$ (W/m2)')
    plt.ylim(ylim)

    plt.subplot(339)
    for _, ds, label, color, ls in runs:
        plt.plot(ds.lw_flux_up, ds.zh, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$L\uparrow$ (W/m2)')
    plt.ylim(ylim)

In [6]:
def plot_profiles_diff(runs):
    """
    Profiles, difference with first run.
    """
    ylim = (0, 17000)
    ref = runs[0][1]

    plt.figure(figsize=(8, 7), layout='constrained')

    plt.subplot(331)
    for _, ds, label, color, ls in runs[1:]:
        plt.plot(ds.thl - ref.thl, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$\Delta\theta_l$ (K)')
    plt.ylabel('z (m)')
    plt.ylim(ylim)

    plt.subplot(332)
    for _, ds, label, color, ls in runs[1:]:
        plt.plot((ds.qt - ref.qt)*1000, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$\Delta q_t$ (g/kg)')
    plt.ylabel('z (m)')
    plt.ylim(ylim)

    plt.subplot(333)
    for _, ds, label, color, ls in runs[1:]:
        plt.plot((ds.ql - ref.ql)*1000, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$\Delta q_l$ (g/kg)')
    plt.ylabel('z (m)')
    plt.ylim(ylim)

    plt.subplot(334)
    for _, ds, label, color, ls in runs[1:]:
        plt.plot((ds.qi - ref.qi)*1000, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$\Delta q_i$ (g/kg)')
    plt.ylabel('z (m)')
    plt.ylim(ylim)

    plt.subplot(335)
    for _, ds, label, color, ls in runs[1:]:
        plt.plot((ds.qr - ref.qr)*1000, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$\Delta q_r$ (g/kg)')
    plt.ylim(ylim)

    plt.subplot(336)
    for _, ds, label, color, ls in runs[1:]:
        plt.plot((ds.qs - ref.qs)*1000, ds.z, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$\Delta q_s$ (g/kg)')
    plt.ylim(ylim)

    plt.subplot(337)
    for _, ds, label, color, ls in runs[1:]:
        plt.plot(ds.sw_flux_dn - ref.sw_flux_dn, ds.zh, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$\Delta S\downarrow$ (W/m2)')
    plt.ylabel('z (m)')
    plt.ylim(ylim)

    plt.subplot(338)
    for _, ds, label, color, ls in runs[1:]:
        plt.plot(ds.lw_flux_dn - ref.lw_flux_dn, ds.zh, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$\Delta L\downarrow$ (W/m2)')
    plt.ylim(ylim)

    plt.subplot(339)
    for _, ds, label, color, ls in runs[1:]:
        plt.plot(ds.lw_flux_up - ref.lw_flux_up, ds.zh, label=label, color=color, ls=ls)
    plt.legend()
    plt.xlabel(r'$\Delta L\uparrow$ (W/m2)')
    plt.ylim(ylim)

## Impact vertical grid reduction

- 144 = Original RCEMIP-I grid.
- 128l = 128 levels, less aggressive stretching above cloud top, domain top ~25 km.
- 128h = 128 levels, more aggressive stretching above cloud top, domain top ~32 km

In [7]:
# RCEMIP grid:
z1 = np.loadtxt('rcemip_les_grid.txt')[:-2]

# 128, +/- same as RCEMIP <15 km, more agressive stretching above.
z = np.array([0, 3_000, 15_000, 100_000])
f = np.array([1.05, 1.00, 1.055])
grid = ls2d.grid.Grid_stretched_manual(128, 40, z, f)
z2 = grid.z

# 128, +/- same as RCEMIP <15 km, less agressive stretching, lower domain.
z = np.array([0, 3_000, 15_000, 100_000])
f = np.array([1.05, 1.00, 1.03])
grid = ls2d.grid.Grid_stretched_manual(128, 40, z, f)
z3 = grid.z

dz1 = z1[1:] - z1[:-1]
dz2 = z2[1:] - z2[:-1]
dz3 = z3[1:] - z3[:-1]

cloud_top = 15_000

plt.figure(layout='constrained')

plt.subplot(121)
plt.plot(z1, '-o', color='k', ms=6, mfc='none', label='n=144 (original RCEMIP')
plt.plot(z3, '-s', color='C0', ms=6, mfc='none', label='n=128, low')
plt.plot(z2, '-^', color='C2', ms=6, mfc='none', label='n=128, high')
plt.axhline(cloud_top, color='tab:red', linestyle='--', label='cloud top')
plt.xlabel('Level (-)')
plt.ylabel('z (m)')
plt.legend()
plt.ylim(0, 32_000)
plt.grid()

plt.subplot(122)
plt.plot(dz1, z1[:-1], '-o', color='k', ms=6, mfc='none')
plt.plot(dz3, z3[:-1], '-s', color='C0', ms=6, mfc='none')
plt.plot(dz2, z2[:-1], '-^', color='C2', ms=6, mfc='none')
plt.axhline(cloud_top, color='tab:red', linestyle='--', label='cloud top')
plt.xlabel('dz (m))')
plt.ylabel('z (m)')
plt.ylim(0, 32_000)
plt.grid()

FileNotFoundError: rcemip_les_grid.txt not found.

In [ ]:
runs_vgrid = (
    [ds_144, dsm_144, r'144-$\Delta$200', 'k', '-'],
    [ds_128, dsm_128, r'128low-$\Delta$200', 'C0', '-'],
    [ds_200, dsm_200, r'128high-$\Delta$200', 'C2', '-'],
)

plot_time_series(runs_vgrid)
plot_profiles(runs_vgrid)

## Impact horizontal resolution

In [ ]:
runs_hgrid = (
    [ds_100, dsm_100, r'128h-$\Delta$100', 'k', '-'],
    [ds_200, dsm_200, r'128h-$\Delta$200', 'C0', '-'],
    [ds_400, dsm_400, r'128h-$\Delta$400', 'C2', '-'],
    [ds_800, dsm_800, r'128h-$\Delta$800', 'C3', '-'],
)

plot_time_series(runs_hgrid)
plot_profiles(runs_hgrid)
plot_profiles_diff(runs_hgrid)